# Engram × CharacterEval

Applies the **CharacterEval** evaluation framework (Tu et al., 2024 — `2401.01275`) to Engram's personality-parameterized NPCs.

CharacterEval defines **13 metrics across 4 dimensions**:

| Dimension | Metrics |
|---|---|
| Conversational Ability | Fluency (Flu.), Coherency (Coh.), Consistency (Cons.) |
| Character Consistency | Knowledge-Exposure (KE), Knowledge-Accuracy (KA), Knowledge-Hallucination (KH), Persona-Behavior (PB), Persona-Utterance (PU) |
| Role-playing Attractiveness | Human-Likeness (HL), Communication Skills (CS), Expression Diversity (ED), Empathy (Emp.) |
| Personality Back-Testing | OCEAN accuracy (adapted from MBTI — Engram uses OCEAN, not MBTI) |

**Adaptations from the original paper:**
- Dataset: Chinese novels → our 6 English events (same for all NPCs)
- Judge: CharacterRM (Chinese Baichuan reward model) → Gemini-as-judge (same approach as paper's GPT-4 variant, Table 5)
- Personality test: MBTI → OCEAN prediction accuracy (more natural for Engram)
- Scale: 1–5 for all 12 subjective metrics; MAE on 0–1 for OCEAN back-test

**Reference GPT-4 scores (CharacterEval Table 5):**
Flu=4.63, Coh=4.85, Cons=4.66, KE=4.92, KA=4.92, KH=4.90, PB=4.91, PU=4.91, HL=4.80, CS=3.95, ED=3.81, Emp=3.88, MBTI=0.694

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'google-genai', 'numpy', 'matplotlib', 'pandas', '-q'])

In [ ]:
# Auto-reload engram source files on every cell execution
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys, json, time, copy, textwrap
from pathlib import Path

project_root = Path('..').resolve()
sys.path.insert(0, str(project_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({'figure.facecolor': '#FAFAFA', 'axes.facecolor': '#FAFAFA'})
print('Ready. Project root:', project_root)

In [ ]:
# API key
API_KEY = os.environ.get('GEMINI_API_KEY', '')
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get('GEMINI_API_KEY')
    except Exception:
        pass
if not API_KEY:
    raise EnvironmentError('Set GEMINI_API_KEY before running.')
os.environ['GEMINI_API_KEY'] = API_KEY
print('API key loaded.')

In [ ]:
from engram.models import OCEANProfile, NPCConfig, Memory
from engram.llm.client import GeminiClient
from engram.npc import NPCAgent

llm = GeminiClient(api_key=API_KEY)
print('Engram + GeminiClient ready.')

## 1. CharacterEval Metric Definitions

Adapted from the paper's annotation guidelines. Each metric is scored 1–5 by Gemini-as-judge.

In [ ]:
# ── 12 subjective metrics (CharacterEval §5) ──────────────────────────────────
METRICS = {
    # Dimension 1: Conversational Ability
    'Flu': {
        'name': 'Fluency',
        'dimension': 'Conversational Ability',
        'description': (
            'Evaluate grammatical correctness and readability. '
            'Is the response free from obvious errors, awkward phrasing, or broken sentences? '
            '1=incomprehensible, 5=perfectly fluent natural English.'
        ),
    },
    'Coh': {
        'name': 'Coherency',
        'dimension': 'Conversational Ability',
        'description': (
            'Evaluate topic relevance between the response and the conversation context. '
            'Does the NPC stay on-topic and respond sensibly to what the player said? '
            '1=completely off-topic, 5=perfectly on-topic and relevant.'
        ),
    },
    'Cons': {
        'name': 'Consistency',
        'dimension': 'Conversational Ability',
        'description': (
            'Evaluate whether this response contradicts the NPC\'s earlier responses in the conversation. '
            'Does the NPC maintain a stable stance and not contradict themselves? '
            '1=directly contradicts prior statements, 5=fully consistent.'
        ),
    },
    # Dimension 2: Character Consistency
    'KE': {
        'name': 'Knowledge-Exposure',
        'dimension': 'Character Consistency',
        'description': (
            'Evaluate whether the NPC demonstrates knowledge appropriate to their character. '
            'Does the NPC reference their backstory, expertise, or world knowledge? '
            '1=no character knowledge shown, 5=rich relevant knowledge demonstrated.'
        ),
    },
    'KA': {
        'name': 'Knowledge-Accuracy',
        'dimension': 'Character Consistency',
        'description': (
            'Evaluate whether the knowledge the NPC expresses is accurate relative to their character profile. '
            'Does the NPC avoid making factual errors about themselves, their world, or other characters? '
            '1=major factual errors, 5=all knowledge accurate.'
        ),
    },
    'KH': {
        'name': 'Knowledge-Hallucination',
        'dimension': 'Character Consistency',
        'description': (
            'Evaluate whether the NPC avoids hallucinating facts inconsistent with their character. '
            'Does the NPC stay within the boundaries of what their character would realistically know? '
            '1=frequent hallucinations/fabrications, 5=no hallucinations.'
        ),
    },
    'PB': {
        'name': 'Persona-Behavior',
        'dimension': 'Character Consistency',
        'description': (
            'Evaluate whether the NPC\'s described actions and behaviors match their personality profile. '
            'A high-Neuroticism NPC should behave anxiously; a high-Agreeableness NPC should be warm. '
            '1=behavior directly contradicts personality, 5=behavior perfectly matches personality.'
        ),
    },
    'PU': {
        'name': 'Persona-Utterance',
        'dimension': 'Character Consistency',
        'description': (
            'Evaluate whether the NPC\'s speaking style and word choice match their personality. '
            'A reserved, anxious NPC should speak differently than a warm social one. '
            '1=speech style contradicts personality, 5=speech perfectly reflects personality.'
        ),
    },
    # Dimension 3: Role-playing Attractiveness
    'HL': {
        'name': 'Human-Likeness',
        'dimension': 'Role-playing Attractiveness',
        'description': (
            'Evaluate how human and natural the NPC sounds, rather than robotic or formulaic. '
            'Does the response feel like something a real person would say? '
            '1=obviously machine-generated and robotic, 5=indistinguishable from a human.'
        ),
    },
    'CS': {
        'name': 'Communication Skills',
        'dimension': 'Role-playing Attractiveness',
        'description': (
            'Evaluate the NPC\'s conversational intelligence and social skill (EQ). '
            'Does the NPC navigate the conversation skillfully, ask good follow-ups, or handle conflict well? '
            '1=poor communication, 5=excellent communicator.'
        ),
    },
    'ED': {
        'name': 'Expression Diversity',
        'dimension': 'Role-playing Attractiveness',
        'description': (
            'Evaluate vocabulary variety and expressive richness. '
            'Does the NPC use diverse, character-appropriate vocabulary rather than repetitive generic phrases? '
            '1=repetitive/generic, 5=richly varied and expressive.'
        ),
    },
    'Emp': {
        'name': 'Empathy',
        'dimension': 'Role-playing Attractiveness',
        'description': (
            'Evaluate whether the NPC recognizes and appropriately responds to the player\'s emotional state. '
            'Does the NPC show awareness of what the player might be feeling? '
            '1=ignores player emotion, 5=strongly empathetic and emotionally aware.'
        ),
    },
}

DIMENSIONS = ['Conversational Ability', 'Character Consistency',
               'Role-playing Attractiveness']
DIM_METRICS = {
    'Conversational Ability': ['Flu', 'Coh', 'Cons'],
    'Character Consistency': ['KE', 'KA', 'KH', 'PB', 'PU'],
    'Role-playing Attractiveness': ['HL', 'CS', 'ED', 'Emp'],
}

print(f'{len(METRICS)} metrics defined across {len(DIMENSIONS)} dimensions.')

## 2. Gemini-as-Judge

Adapted from CharacterEval's GPT-4 evaluation (Table 5). Input format mirrors CharacterRM's `<RoleInfo> / <Context> / <Response> / <Dimension>` structure.

In [ ]:
def judge_response(role_info: str, context: str, response: str,
                   metric_key: str, llm: GeminiClient) -> float:
    """
    Score one NPC response on one metric using Gemini-as-judge.
    Returns a float on the 1–5 scale.
    Mirrors CharacterEval's CharacterRM input format.
    """
    m = METRICS[metric_key]
    prompt = f"""You are an expert evaluator for role-playing conversational agents.
Score the following NPC response on the metric described below. Return ONLY a JSON object: {{"score": <int 1-5>, "reason": "<one sentence>"}}

<RoleInfo>
{role_info}

<Context>
{context}

<Response>
{response}

<Metric: {m['name']}>
{m['description']}

Return ONLY valid JSON with "score" (1-5 integer) and "reason" (one sentence)."""

    result = llm.generate_json(prompt)
    score = result.get('score', 3)
    try:
        return float(int(score))
    except (TypeError, ValueError):
        return 3.0


def ocean_backtest(role_info: str, full_dialogue: str,
                   true_profile: OCEANProfile, llm: GeminiClient) -> dict:
    """
    OCEAN personality back-test (adapted from CharacterEval's MBTI back-test).
    Ask Gemini to predict the NPC's OCEAN values from the dialogue alone,
    then compute MAE and per-trait accuracy (within ±0.2 = correct).
    """
    prompt = f"""You are a personality psychologist analyzing a conversational transcript.
Based ONLY on the NPC's responses in the dialogue below (not the role info), 
estimate this NPC's OCEAN Big Five personality scores on a 0.0–1.0 scale.

Return ONLY JSON: {{"O": <float>, "C": <float>, "E": <float>, "A": <float>, "N": <float>, "reasoning": "<two sentences>"}}

<RoleInfo (for context only — do not use for scoring)>
{role_info}

<Dialogue>
{full_dialogue}

Return ONLY valid JSON with O, C, E, A, N (each 0.0–1.0) and reasoning."""

    result = llm.generate_json(prompt)
    traits = ('O', 'C', 'E', 'A', 'N')
    true_vals = {'O': true_profile.O, 'C': true_profile.C, 'E': true_profile.E,
                 'A': true_profile.A, 'N': true_profile.N}
    
    pred_vals = {}
    for t in traits:
        try:
            pred_vals[t] = max(0.0, min(1.0, float(result.get(t, 0.5))))
        except (TypeError, ValueError):
            pred_vals[t] = 0.5
    
    mae = np.mean([abs(pred_vals[t] - true_vals[t]) for t in traits])
    # Accuracy: each trait correct if |pred - true| <= 0.2 (analogue to MBTI accuracy)
    per_trait_correct = {t: abs(pred_vals[t] - true_vals[t]) <= 0.2 for t in traits}
    accuracy = np.mean(list(per_trait_correct.values()))
    
    return {
        'predicted': pred_vals,
        'true': true_vals,
        'mae': round(float(mae), 4),
        'accuracy': round(float(accuracy), 4),
        'per_trait_correct': per_trait_correct,
        'reasoning': result.get('reasoning', ''),
    }

print('Judge functions defined.')

## 3. NPC Setup (same 3 profiles, same 6 events as Engram §4)

In [ ]:
PERSONA = (
    "You are a townsperson in a small 17th-century English port town. "
    "You work at the dockside storeroom and manage local goods. "
    "You know the mayor, local merchant, and most residents by name. "
    "You have lived here your whole life and take your responsibilities seriously."
)

BACKSTORY = [
    "You grew up in this town and have worked the docks since you were twelve.",
    "The mayor has always seemed trustworthy and organized town affairs well.",
    "Strange things sometimes wash ashore — you have learned to be cautious.",
    "The harvest festival is the most important community event of the year.",
    "You keep the storeroom key on you at all times — it is your responsibility.",
    "Sickness travels fast in a port town; you have lost people to disease before.",
]

PROFILES = [
    ('guard',    'Paranoid Guard',    OCEANProfile('Paranoid Guard',    O=0.2, C=0.5, E=0.3, A=0.2, N=0.9)),
    ('merchant', 'Friendly Merchant', OCEANProfile('Friendly Merchant', O=0.5, C=0.5, E=0.9, A=0.8, N=0.2)),
    ('clerk',    'Rigid Clerk',       OCEANProfile('Rigid Clerk',       O=0.1, C=0.9, E=0.3, A=0.5, N=0.4)),
]

EVENTS = [
    {'name': 'Physical Threat',
     'turns': ["Give me the storeroom key. Now. I know you have it.",
               "Don't make this difficult. Hand it over or things will get unpleasant."]},
    {'name': 'Social Opportunity',
     'turns': ["Hey! We're all getting together for the harvest dinner tonight. You should come!",
               "It'll be a great time — the whole town will be there. Music, food, dancing."]},
    {'name': 'Contradictory Info',
     'turns': ["I have evidence that Mayor Fletcher has been embezzling from the town fund for months.",
               "I've seen the ledgers myself. The numbers don't lie. What do you think we should do?"]},
    {'name': 'Cooperative Request',
     'turns': ["Excuse me — my child is very sick and I can't afford a doctor. Could you help?",
               "Anything you can spare would mean the world. She has a fever and it's getting worse."]},
    {'name': 'Goal-Relevant Task',
     'turns': ["The morning shipment arrived two hours early. Can you process it before market opens?",
               "There are twelve crates. Can you get started right away?"]},
    {'name': 'Ambiguous Stimulus',
     'turns': ["There's a strange animal loose in the town square. Nobody knows what it is.",
               "It hasn't attacked anyone but people are frightened. Some say it's a bad omen."]},
]

print(f'{len(PROFILES)} NPCs × {len(EVENTS)} events × 2 turns = {len(PROFILES)*len(EVENTS)*2} total turns')

In [ ]:
import shutil

EVAL_DIR = str(project_root / 'eval' / 'chareval_data')
shutil.rmtree(EVAL_DIR, ignore_errors=True)
os.makedirs(EVAL_DIR, exist_ok=True)

agents = {}
for npc_id, label, profile in PROFILES:
    config = NPCConfig(npc_id=npc_id, name=label, persona=PERSONA,
                       backstory=BACKSTORY, profile=profile)
    print(f'Initializing {label}...', end=' ', flush=True)
    agent = NPCAgent(config, llm, data_dir=EVAL_DIR)
    agents[npc_id] = (label, agent)
    print(f'{len(agent.memory_manager.all_memories)} backstory memories.')

## 4. Collect Dialogue Data

Run all events, capture every (context, response) pair for scoring.

In [ ]:
# Force-reload fixed engram modules so the kernel picks up source changes
# without requiring a full kernel restart.
import importlib, sys

for _mod in [
    'engram.pipeline.consolidation',
    'engram.pipeline.threat',
    'engram.pipeline.retrieval',
    'engram.pipeline.response',
    'engram.memory.manager',
    'engram.memory.keystore',
    'engram.npc',
]:
    if _mod in sys.modules:
        importlib.reload(sys.modules[_mod])

# Patch the function reference in npc's module namespace (import-by-name
# bindings don't update automatically when a dependency is reloaded).
import engram.pipeline.consolidation as _cons
import engram.npc as _npc_mod
_npc_mod.post_session_fact_check = _cons.post_session_fact_check

print("engram modules reloaded — safe to run end_session")

In [ ]:
# dialogue_data[npc_id] = list of {event, turn_idx, context, response, player_input}
dialogue_data = {npc_id: [] for npc_id, _, _ in PROFILES}

for event in EVENTS:
    print(f"\n── {event['name']} ──")
    for t_idx, player_input in enumerate(event['turns']):
        print(f"  Player: {player_input[:70]}")
        for npc_id, (label, agent) in agents.items():
            # Snapshot context before the response
            context_snapshot = '\n'.join(
                f"Player: {h['player']}\nNPC: {h['npc']}" 
                for h in agent.history[-4:]
            ) or '(Start of conversation)'
            
            response = agent.run_turn(player_input)
            
            dialogue_data[npc_id].append({
                'event': event['name'],
                'turn_idx': t_idx,
                'player_input': player_input,
                'context': context_snapshot,
                'response': response,
            })
            print(f"  [{label[:16]:16s}]: {response[:80]}")
        time.sleep(1)

print('\n✅ All dialogues collected.')

In [ ]:
# End sessions
for npc_id, (label, agent) in agents.items():
    agent.end_session()
    print(f'Session ended: {label}')

## 5. Score All Responses — Gemini as Judge

12 metrics × N responses × 3 NPCs. This is the most expensive cell — ~36 × 12 = 432 LLM calls. Takes a few minutes.

In [ ]:
def build_role_info(npc_id: str) -> str:
    _, agent = agents[npc_id]
    p = agent.config.profile
    return (
        f"Name: {agent.config.name}\n"
        f"Personality: {p.describe()}\n"
        f"  O={p.O} (Openness), C={p.C} (Conscientiousness), E={p.E} (Extraversion), "
        f"A={p.A} (Agreeableness), N={p.N} (Neuroticism)\n"
        f"Background: {agent.config.persona}\n"
        f"Backstory: {' | '.join(agent.config.backstory)}"
    )

# scores[npc_id][metric_key] = list of float scores
scores = {npc_id: {m: [] for m in METRICS} for npc_id, _, _ in PROFILES}

for npc_id, label, _ in PROFILES:
    role_info = build_role_info(npc_id)
    records = dialogue_data[npc_id]
    n = len(records)
    print(f"\nScoring {label} ({n} responses × {len(METRICS)} metrics = {n*len(METRICS)} calls)")
    
    for i, record in enumerate(records):
        context = f"{record['context']}\nPlayer: {record['player_input']}"
        response = record['response']
        
        for metric_key in METRICS:
            score = judge_response(role_info, context, response, metric_key, llm)
            scores[npc_id][metric_key].append(score)
        
        if (i + 1) % 4 == 0:
            print(f"  {i+1}/{n} responses scored")
        time.sleep(0.3)  # rate limiting

print('\n✅ All responses scored.')

## 6. OCEAN Back-Testing

Adapted from CharacterEval's MBTI back-test. Gemini reads the full dialogue transcript and predicts OCEAN values. Accuracy = fraction of traits predicted within ±0.2.

In [ ]:
backtest_results = {}

for npc_id, label, profile in PROFILES:
    role_info = build_role_info(npc_id)
    
    # Build full dialogue transcript
    lines = []
    for record in dialogue_data[npc_id]:
        lines.append(f"[Event: {record['event']}]")
        lines.append(f"Player: {record['player_input']}")
        lines.append(f"NPC: {record['response']}")
        lines.append('')
    full_dialogue = '\n'.join(lines)
    
    print(f'Running OCEAN back-test for {label}...', end=' ', flush=True)
    result = ocean_backtest(role_info, full_dialogue, profile, llm)
    backtest_results[npc_id] = result
    print(f"accuracy={result['accuracy']:.3f}  MAE={result['mae']:.3f}")
    print(f"  True:  " + '  '.join(f"{t}={result['true'][t]:.1f}" for t in 'OCEAN'))
    print(f"  Pred:  " + '  '.join(f"{t}={result['predicted'][t]:.2f}" for t in 'OCEAN'))

print('\n✅ OCEAN back-tests complete.')

## 7. Results Table — CharacterEval Format

Matches Table 4 layout from the paper (CharacterRM scores). Reference: GPT-4 row from Table 5.

In [ ]:
# Compute averages
avg_scores = {}
for npc_id, label, _ in PROFILES:
    avg_scores[npc_id] = {m: round(np.mean(scores[npc_id][m]), 3) for m in METRICS}

# Build Table 4-style output
METRIC_ABBREVS = ['Flu', 'Coh', 'Cons', 'KE', 'KA', 'KH', 'PB', 'PU', 'HL', 'CS', 'ED', 'Emp']

rows = []
for npc_id, label, profile in PROFILES:
    row = {'Model': label}
    for dim, mkeys in DIM_METRICS.items():
        for m in mkeys:
            row[m] = avg_scores[npc_id][m]
        row[f'{dim[:4]} Avg'] = round(np.mean([avg_scores[npc_id][m] for m in mkeys]), 3)
    row['OCEAN Acc'] = round(backtest_results[npc_id]['accuracy'], 3)
    rows.append(row)

# GPT-4 reference from CharacterEval Table 5
gpt4_row = {
    'Model': 'GPT-4 (CharacterEval ref)',
    'Flu': 4.630, 'Coh': 4.850, 'Cons': 4.656,
    'Conv Avg': 4.712,
    'KE': 4.924, 'KA': 4.923, 'KH': 4.899, 'PB': 4.912, 'PU': 4.906,
    'Char Avg': 4.913,
    'HL': 4.796, 'CS': 3.947, 'ED': 3.806, 'Emp': 3.883,
    'Role Avg': 4.108,
    'OCEAN Acc': 0.694,  # was MBTI in original
}
rows.append(gpt4_row)

df = pd.DataFrame(rows).set_index('Model')
print('\n── CharacterEval Results — Engram NPCs (Gemini-as-judge, 1–5 scale) ──')
print(df.to_string())
print('\nNote: GPT-4 reference is Chinese role-playing chars; our NPCs are English — not directly comparable, shown for scale context.')

In [ ]:
# Compact summary table
summary_rows = []
for npc_id, label, _ in PROFILES:
    s = avg_scores[npc_id]
    summary_rows.append({
        'NPC': label,
        'Conv. Ability': round(np.mean([s[m] for m in ['Flu','Coh','Cons']]), 2),
        'Char. Consistency': round(np.mean([s[m] for m in ['KE','KA','KH','PB','PU']]), 2),
        'Role Attractiveness': round(np.mean([s[m] for m in ['HL','CS','ED','Emp']]), 2),
        'OCEAN Acc.': round(backtest_results[npc_id]['accuracy'], 3),
        'OCEAN MAE': round(backtest_results[npc_id]['mae'], 3),
    })

df_summary = pd.DataFrame(summary_rows).set_index('NPC')
print('\n── Dimension Averages ──')
print(df_summary.to_string())

## 8. Visualizations

In [ ]:
# ── Fig 1: 12-metric heatmap ──────────────────────────────────────────────────
import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=(13, 3.5))
colors = {'guard': '#DC2626', 'merchant': '#0D9488', 'clerk': '#7C3AED'}

npc_ids = [nid for nid, _, _ in PROFILES]
labels = [lbl for _, lbl, _ in PROFILES]
matrix = np.array([[avg_scores[npc_id][m] for m in METRIC_ABBREVS] for npc_id in npc_ids])

im = ax.imshow(matrix, cmap='RdYlGn', vmin=1, vmax=5, aspect='auto')
ax.set_xticks(range(len(METRIC_ABBREVS)))
ax.set_xticklabels(METRIC_ABBREVS, fontsize=11)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=11)

# Add dimension separators
for x in [2.5, 7.5]:
    ax.axvline(x, color='white', linewidth=2)

# Add value annotations
for i in range(len(npc_ids)):
    for j in range(len(METRIC_ABBREVS)):
        ax.text(j, i, f'{matrix[i,j]:.2f}', ha='center', va='center',
                fontsize=9, color='black')

plt.colorbar(im, ax=ax, label='Score (1–5)')
ax.set_title('CharacterEval Scores per NPC — 12 Metrics × 3 Personalities', fontweight='bold', pad=10)

# Dimension labels above
for dim, mkeys, xc in [('Conv. Ability', ['Flu','Coh','Cons'], 1),
                        ('Character Consistency', ['KE','KA','KH','PB','PU'], 5),
                        ('Role-playing Attract.', ['HL','CS','ED','Emp'], 10)]:
    ax.text(xc, -0.8, dim, ha='center', va='bottom', fontsize=9,
            color='#374151', fontweight='bold',
            transform=ax.get_xaxis_transform())

plt.tight_layout()
plt.savefig('chareval_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chareval_heatmap.png')

In [ ]:
# ── Fig 2: Radar chart — 4 dimensions ────────────────────────────────────────
dim_avgs = {}
for npc_id, label, _ in PROFILES:
    s = avg_scores[npc_id]
    dim_avgs[npc_id] = [
        np.mean([s[m] for m in ['Flu','Coh','Cons']]),
        np.mean([s[m] for m in ['KE','KA','KH','PB','PU']]),
        np.mean([s[m] for m in ['HL','CS','ED','Emp']]),
        backtest_results[npc_id]['accuracy'] * 5,  # scale 0-1 → 0-5
    ]

dim_labels = ['Conversational\nAbility', 'Character\nConsistency',
               'Role-playing\nAttract.', 'OCEAN\nBack-test (×5)']
N = len(dim_labels)
angles = [n / N * 2 * np.pi for n in range(N)] + [0]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
npc_colors = {'guard': '#DC2626', 'merchant': '#0D9488', 'clerk': '#7C3AED'}

for npc_id, label, _ in PROFILES:
    vals = dim_avgs[npc_id] + [dim_avgs[npc_id][0]]
    ax.plot(angles, vals, 'o-', linewidth=2, label=label, color=npc_colors[npc_id])
    ax.fill(angles, vals, alpha=0.1, color=npc_colors[npc_id])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(dim_labels, fontsize=10)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1','2','3','4','5'], fontsize=8)
ax.set_title('CharacterEval Dimension Scores\n(Engram NPCs, Gemini judge)',
             fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)
plt.tight_layout()
plt.savefig('chareval_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chareval_radar.png')

In [ ]:
# ── Fig 3: OCEAN back-test — predicted vs true per NPC ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
trait_names = list('OCEAN')
x = np.arange(5)
w = 0.35

for ax, (npc_id, label, _) in zip(axes, PROFILES):
    bt = backtest_results[npc_id]
    true_v = [bt['true'][t] for t in trait_names]
    pred_v = [bt['predicted'][t] for t in trait_names]
    
    ax.bar(x - w/2, true_v, w, label='True', color=npc_colors[npc_id], alpha=0.85)
    ax.bar(x + w/2, pred_v, w, label='Predicted', color='#94A3B8', alpha=0.8,
           edgecolor='black', linewidth=0.7)
    
    ax.set_xticks(x)
    ax.set_xticklabels(trait_names, fontsize=12)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('OCEAN value (0–1)')
    ax.set_title(f"{label}\nAcc={bt['accuracy']:.2f}  MAE={bt['mae']:.3f}",
                 fontweight='bold', color=npc_colors[npc_id])
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.2)
    
    # Mark correct/incorrect
    for i, t in enumerate(trait_names):
        correct = bt['per_trait_correct'][t]
        ax.text(i, 1.08, '✓' if correct else '✗', ha='center', fontsize=14,
                color='#16A34A' if correct else '#DC2626')

plt.suptitle('OCEAN Back-Test: Gemini predicts personality from dialogue alone (±0.2 = correct)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('chareval_ocean_backtest.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chareval_ocean_backtest.png')

In [ ]:
# ── Fig 4: Per-metric scores per NPC (grouped bar) ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, (npc_id, label, _) in zip(axes, PROFILES):
    s = avg_scores[npc_id]
    vals = [s[m] for m in METRIC_ABBREVS]
    bar_colors = (['#3B82F6']*3 + ['#F59E0B']*5 + ['#10B981']*4)
    ax.barh(METRIC_ABBREVS[::-1], vals[::-1], color=bar_colors[::-1], alpha=0.85, edgecolor='white')
    ax.set_xlim(1, 5)
    ax.axvline(3.0, color='red', linestyle=':', linewidth=1, alpha=0.5, label='midpoint')
    ax.set_title(label, fontweight='bold', color=npc_colors[npc_id])
    ax.set_xlabel('Score (1–5)')
    ax.grid(axis='x', alpha=0.2)
    for i, v in enumerate(vals[::-1]):
        ax.text(v + 0.02, i, f'{v:.2f}', va='center', fontsize=9)

# Legend patches for dimensions
legend_patches = [
    mpatches.Patch(color='#3B82F6', label='Conv. Ability'),
    mpatches.Patch(color='#F59E0B', label='Char. Consistency'),
    mpatches.Patch(color='#10B981', label='Role Attractiveness'),
]
axes[0].legend(handles=legend_patches, fontsize=8, loc='lower right')
plt.suptitle('CharacterEval — Per-Metric Scores (1–5, Gemini judge)', fontweight='bold')
plt.tight_layout()
plt.savefig('chareval_per_metric.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chareval_per_metric.png')

## 9. Final Summary

In [ ]:
print('=' * 72)
print('ENGRAM × CHARACTEREVAL — FINAL RESULTS')
print('=' * 72)
print(f"{'Metric':<8}" + ''.join(f"{lbl[:16]:>18}" for _, lbl, _ in PROFILES))
print('-' * 62)
for m in METRIC_ABBREVS:
    row = f"{m:<8}" + ''.join(f"{avg_scores[nid][m]:>18.3f}" for nid, _, _ in PROFILES)
    print(row)
print('-' * 62)
for dim, mkeys in DIM_METRICS.items():
    row = f"{dim[:8]:<8}" + ''.join(
        f"{np.mean([avg_scores[nid][m] for m in mkeys]):>18.3f}"
        for nid, _, _ in PROFILES
    )
    print(row + '  (dim avg)')
print('-' * 62)
print(f"{'OCEAN':<8}" + ''.join(
    f"{backtest_results[nid]['accuracy']:>18.3f}" for nid, _, _ in PROFILES
) + '  (back-test acc)')
print(f"{'MAE':<8}" + ''.join(
    f"{backtest_results[nid]['mae']:>18.4f}" for nid, _, _ in PROFILES
) + '  (back-test MAE)')

# Save
output = {
    'metric_scores': {nid: avg_scores[nid] for nid, _, _ in PROFILES},
    'dimension_avgs': {
        nid: {dim: round(np.mean([avg_scores[nid][m] for m in mkeys]), 3)
              for dim, mkeys in DIM_METRICS.items()}
        for nid, _, _ in PROFILES
    },
    'ocean_backtest': {nid: {
        'accuracy': backtest_results[nid]['accuracy'],
        'mae': backtest_results[nid]['mae'],
        'predicted': backtest_results[nid]['predicted'],
        'true': backtest_results[nid]['true'],
    } for nid, _, _ in PROFILES},
}
with open(project_root / 'eval' / 'chareval_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print('\nSaved: eval/chareval_results.json')

## Interpretation Notes

**Why scores may differ from CharacterEval Table 5 (GPT-4 baseline):**
- CharacterEval uses Chinese cultural characters from novels; our NPCs are English historical fiction
- CharacterRM is trained on Chinese role-playing annotations; Gemini-as-judge is zero-shot
- Gemini exhibits well-documented leniency bias (inflates scores vs human judgment)

**What to look for:**
- **PB/PU (Persona-Behavior/Utterance)**: Should differ across personalities — the key Engram claim is that personality shapes HOW the NPC behaves, not just what it says
- **OCEAN back-test accuracy**: Can Gemini correctly infer the personality from dialogue alone? High accuracy validates that personality is legible in the output
- **CS/Emp (Communication Skills/Empathy)**: Should be highest for Friendly Merchant (high-A, high-E), lowest for Paranoid Guard
- **Cons (Consistency)**: Should be highest for Rigid Clerk (high-C, low-O = very consistent beliefs)